# Supplementary1a autoreactivity


In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

H5AD_PATH = "data/adata_cohort1.h5ad"
LAYER     = "zscores_4andhalf"
GROUP_COL = "case_control"

OUTPUT_DIR = "results/fig"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

PALETTE = {"HC": "#4682B4", "SLE": "#FC9E80"}
STRIP_KW = dict(size=1.8, alpha=0.35, jitter=True, dodge=False, linewidth=0, color=".2")

In [ ]:
print("Loading AnnData...")
adata = ad.read_h5ad(H5AD_PATH)
print(f"  {adata.shape[0]} samples x {adata.shape[1]} peptides")
print(adata.obs[GROUP_COL].value_counts(dropna=False))

X = adata.layers[LAYER]
print(f"  layer '{LAYER}': dtype={X.dtype}, shape={X.shape}")

group_map = {"lupus": "SLE", "control": "HC"}
group = adata.obs[GROUP_COL].map(group_map)

missing = group.isna()
if missing.any():
    recovered = adata.obs.loc[missing, "group"].str.startswith(("flare", "clues")).map({True: "SLE", False: pd.NA})
    print(f"  Recovering {missing.sum()} samples with missing '{GROUP_COL}' via 'group' column:")
    print(adata.obs.loc[missing, ["subject_id", "group"]])
    group.loc[missing] = recovered

still_missing = group.isna()
if still_missing.any():
    print(f"  Dropping {still_missing.sum()} samples with unresolved group label")
    adata = adata[~still_missing.values].copy()
    X = adata.layers[LAYER]
    group = group[~still_missing.values]

group = group.values
assert pd.isna(group).sum() == 0, "unmapped group labels found"

In [ ]:
mean_z = X.mean(axis=1)

df = pd.DataFrame({
    "sample": adata.obs_names,
    "group": group,
    "mean_z": mean_z,
})

print(df.groupby("group")[["mean_z"]].agg(["mean", "median", "std"]))

In [ ]:
def mwu_report(df, col, label):
    sle = df.loc[df["group"] == "SLE", col].values
    hc  = df.loc[df["group"] == "HC",  col].values
    U, p = mannwhitneyu(sle, hc, alternative="two-sided")
    n1, n2 = len(sle), len(hc)
    print(f"[{label}] SLE n={n1} (mean={sle.mean():.3f}, median={np.median(sle):.3f}) "
          f"vs HC n={n2} (mean={hc.mean():.3f}, median={np.median(hc):.3f})")
    print(f"  Mann-Whitney U={U:.1f}, p={p:.3e}")
    return dict(label=label, U=U, p=p, n_sle=n1, n_hc=n2)

res_mean  = mwu_report(df, "mean_z",       "Panel A: per-sample mean z-score (threshold-free)")

In [ ]:
def sig_str(p):
    if p < 0.0001: return "****"
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return "ns"

order = ["SLE", "HC"]

panel_specs = [
    ("mean_z", "Mean z-score (all 360,920 peptides)", res_mean,
     "autoreactivity_mean_z_sle_vs_hc"),
]

for col, ylabel, res, out_stem in panel_specs:
    fig, ax = plt.subplots(figsize=(4.5, 5.5))

    sns.violinplot(data=df, x="group", y=col, order=order, palette=PALETTE,
                    inner=None, linewidth=0.8, cut=0, ax=ax)
    sns.stripplot(data=df, x="group", y=col, order=order,
                  **STRIP_KW, ax=ax)

    n_counts = df.groupby("group").size()
    ymin, ymax = ax.get_ylim()
    yr = ymax - ymin
    for xi, g in enumerate(order):
        ax.text(xi, ymin + 0.02 * yr, f"n={n_counts[g]}",
                ha="center", va="bottom", fontsize=8, color="#555")

    y_bar = ymax - 0.05 * yr
    ax.plot([0, 0, 1, 1], [y_bar, y_bar + 0.03*yr, y_bar + 0.03*yr, y_bar],
            lw=1.0, c="black")
    ax.text(0.5, y_bar + 0.035*yr,
            f"{sig_str(res['p'])}  (p={res['p']:.2e})",
            ha="center", va="bottom", fontsize=8)
    ax.set_ylim(ymin, ymax + 0.15*yr)

    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=10)
    sns.despine(ax=ax)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{out_stem}.pdf", bbox_inches="tight")
    plt.savefig(f"{OUTPUT_DIR}/{out_stem}.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {OUTPUT_DIR}/{out_stem}.pdf")